# 03 — Exposure Labels v2

Identifies anchor posts and classifies users as exposed or unexposed using correct thread-level linking via `link_id`.

**Anchor post definition:**
- Post is in the anchor period: Sep 1–Nov 30, 2023 (cycle 1) or Sep 1–Nov 30, 2024 (cycle 2)
- Post matches negative keyword list (rejection language, re-applicant discourse, anxiety/stress/depression terms)
- Post `mean_mh_score > 0.45` from SVM classifiers (notebook 02)

**Exposed user**: commented on an anchor post thread (identified via `link_id`); anchor post authors excluded

**Unexposed user**: active in r/gradadmissions during Aug 1–May 31 of that cycle but never commented on an anchor thread

**Inputs:**
- `data/processed_v2/posts_clean.jsonl` + `comments_clean.jsonl` (from notebook 01)
- `models/clf_anxiety.joblib`, `clf_depression.joblib`, `clf_stress.joblib` (from notebook 02)

**Outputs:**
- `data/processed_v2/anchor_posts_v2.parquet`
- `data/processed_v2/exposure_labels_v2.parquet` — `author, exposed (bool), cycle`

In [9]:
import json
import pandas as pd
import numpy as np
import joblib
import re
from pathlib import Path

ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'data' / 'processed_v2'
MODEL_DIR = ROOT / 'models'

POSTS_PATH    = DATA_DIR / 'posts_clean.jsonl'
COMMENTS_PATH = DATA_DIR / 'comments_clean.jsonl'

# Cycle windows
CYCLES = {
    1: {
        'anchor_start': '2023-09-01',
        'anchor_end':   '2023-11-30',
        'active_start': '2023-08-01',
        'active_end':   '2024-05-31',
    },
    2: {
        'anchor_start': '2024-09-01',
        'anchor_end':   '2024-11-30',
        'active_start': '2024-08-01',
        'active_end':   '2025-05-31',
    },
}

MH_SCORE_THRESHOLD = 0.45


def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


print('Paths:')
print(' Posts:   ', POSTS_PATH)
print(' Comments:', COMMENTS_PATH)
print(' Models:  ', MODEL_DIR)
print(' Output:  ', DATA_DIR)

Paths:
 Posts:    /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/posts_clean.jsonl
 Comments: /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/comments_clean.jsonl
 Models:   /media/ayush/F/Coding/CS598_Research_Project/models
 Output:   /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2


## 1) Load raw posts

In [10]:
posts = pd.DataFrame(load_jsonl(POSTS_PATH))
posts['created_dt'] = pd.to_datetime(posts['created_dt'], utc=True)
print(f'Clean posts loaded: {len(posts):,} from {posts["author"].nunique():,} unique authors')
print(f'Date range: {posts["created_dt"].min().date()} → {posts["created_dt"].max().date()}')

Clean posts loaded: 78,961 from 41,637 unique authors
Date range: 2023-08-01 → 2025-07-30


## 2) Filter to anchor periods and score with SVM classifiers

In [11]:
# Tag each post with its cycle (if in an anchor period)
def assign_cycle(dt):
    for cycle, w in CYCLES.items():
        if pd.Timestamp(w['anchor_start'], tz='UTC') <= dt <= pd.Timestamp(w['anchor_end'] + ' 23:59:59', tz='UTC'):
            return cycle
    return None

posts['cycle'] = posts['created_dt'].apply(assign_cycle)
anchor_candidates = posts[posts['cycle'].notna()].copy()

print(f'Posts in anchor periods: {len(anchor_candidates):,}')
print(anchor_candidates['cycle'].value_counts().sort_index())

Posts in anchor periods: 14,040
cycle
1.0    7009
2.0    7031
Name: count, dtype: int64


In [12]:
# Load SVM classifiers
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

texts = anchor_candidates['clean_text'].tolist()

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

anchor_candidates['anx_score'] = sigmoid(clf_anx.decision_function(texts))
anchor_candidates['dep_score'] = sigmoid(clf_dep.decision_function(texts))
anchor_candidates['str_score'] = sigmoid(clf_str.decision_function(texts))
anchor_candidates['mean_mh_score'] = anchor_candidates[['anx_score', 'dep_score', 'str_score']].mean(axis=1)

print(f'Scored {len(anchor_candidates):,} anchor-period posts')
print(anchor_candidates['mean_mh_score'].describe().round(4))

Classifiers loaded.
Scored 14,040 anchor-period posts
count    14040.0000
mean         0.3557
std          0.0851
min          0.0871
25%          0.2974
50%          0.3492
75%          0.4072
max          0.7485
Name: mean_mh_score, dtype: float64


## 3) Apply keyword filter → anchor posts

In [13]:
NEGATIVE_KEYWORDS = [
    r'\breject(?:ed|ion)\b',
    r'\bdeclin(?:ed|ing)\b',
    r'\bwaitlist(?:ed)?\b',
    r'\bfunding\s+(?:lost|cut|removed|denied|gap|issue)\b',
    r'\bno\s+funding\b',
    r'\bstipend\b',
    r'\bwithdrew?\s+(?:offer|admission)\b',
    r'\bacceptance\s+rate\b',
    r'\bno\s+(?:offer|response|interview)\b',
    r'\bsilence\s+from\b',
    r'\bnot\s+(?:accepted|admitted|selected)\b',
    r'\bgave\s+up\b',
    r'\bmental\s+health\b',
    r'\banxi(?:ous|ety)\b',
    r'\bdepress(?:ed|ing|ion)\b',
    r'\bstress(?:ed|ful)?\b',
    r'\boverwhelm(?:ed|ing)\b',
    r'\bscared\b',
    r'\bworr(?:ied|ying)\b',
    r'\bfalling\s+apart\b',
    r'\bbreaking\s+down\b',
    r'\bcan(?:\'t|not)\s+(?:take|handle|cope)\b',
    r'\bno\s+chance\b',
    r'\bnot\s+good\s+enough\b',
    r'\bgave\s+up\b',
    r'\bregret\b',
    r'\bfailed\b',
    r'\bimposter\b',
]

keyword_pattern = re.compile('|'.join(NEGATIVE_KEYWORDS), re.IGNORECASE)

anchor_candidates['has_neg_keyword'] = anchor_candidates['clean_text'].str.contains(
    keyword_pattern, na=False
)

anchor_posts = anchor_candidates[
    anchor_candidates['has_neg_keyword'] &
    (anchor_candidates['mean_mh_score'] > MH_SCORE_THRESHOLD)
].copy()

print(f'Anchor posts identified: {len(anchor_posts):,}')
print(anchor_posts['cycle'].value_counts().sort_index())
print(f'Unique anchor authors: {anchor_posts["author"].nunique():,}')
print(f'Mean mh_score of anchor posts: {anchor_posts["mean_mh_score"].mean():.4f}')

Anchor posts identified: 622
cycle
1.0    281
2.0    341
Name: count, dtype: int64
Unique anchor authors: 573
Mean mh_score of anchor posts: 0.5253


In [14]:
# Save anchor posts
anchor_posts[[
    'id', 'author', 'created_dt', 'cycle', 'clean_text',
    'anx_score', 'dep_score', 'str_score', 'mean_mh_score', 'score', 'num_comments'
]].to_parquet(DATA_DIR / 'anchor_posts_v2.parquet', index=False)
print('Saved anchor_posts_v2.parquet')

# Anchor post ID sets per cycle
anchor_ids_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['id'])
    for cycle in [1, 2]
}
print(f'Anchor IDs — cycle 1: {len(anchor_ids_by_cycle[1]):,}, cycle 2: {len(anchor_ids_by_cycle[2]):,}')

# Anchor authors per cycle (to exclude from exposed set)
anchor_authors_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['author'])
    for cycle in [1, 2]
}

Saved anchor_posts_v2.parquet
Anchor IDs — cycle 1: 281, cycle 2: 341


## 4) Load comments → identify exposed users via `link_id`

In [15]:
comments = pd.DataFrame(load_jsonl(COMMENTS_PATH))
comments['created_dt'] = pd.to_datetime(comments['created_dt'], utc=True)
# post_id is already derived from link_id by notebook 01
print(f'Clean comments loaded: {len(comments):,} from {comments["author"].nunique():,} unique authors')

Clean comments loaded: 467,986 from 79,573 unique authors


In [16]:
# All anchor post IDs across both cycles
all_anchor_ids = anchor_ids_by_cycle[1] | anchor_ids_by_cycle[2]

# Comments on anchor posts
anchor_comments = comments[comments['post_id'].isin(all_anchor_ids)].copy()
print(f'Comments on anchor posts: {len(anchor_comments):,}')

# Tag which cycle each anchor comment belongs to
def comment_cycle(post_id):
    if post_id in anchor_ids_by_cycle[1]: return 1
    if post_id in anchor_ids_by_cycle[2]: return 2
    return None

anchor_comments['cycle'] = anchor_comments['post_id'].apply(comment_cycle)
print(anchor_comments['cycle'].value_counts().sort_index())

Comments on anchor posts: 4,861
cycle
1    1937
2    2924
Name: count, dtype: int64


In [17]:
# Exposed users per cycle: commenters on anchor posts, excluding the anchor post authors themselves
exposed_records = []

for cycle in [1, 2]:
    cycle_comments = anchor_comments[anchor_comments['cycle'] == cycle]
    excluded = anchor_authors_by_cycle[cycle]
    exposed_authors = set(cycle_comments['author']) - excluded
    for author in exposed_authors:
        exposed_records.append({'author': author, 'exposed': True, 'cycle': cycle})
    print(f'Cycle {cycle} — exposed users: {len(exposed_authors):,} '
          f'(excluded {len(set(cycle_comments["author"]) & excluded):,} anchor authors)')

exposed_df = pd.DataFrame(exposed_records)
print(f'\nTotal exposed: {len(exposed_df):,}')

Cycle 1 — exposed users: 866 (excluded 111 anchor authors)
Cycle 2 — exposed users: 1,176 (excluded 168 anchor authors)

Total exposed: 2,042


## 5) Identify active users per cycle → unexposed

In [18]:
unexposed_records = []

for cycle, w in CYCLES.items():
    active_start = pd.Timestamp(w['active_start'], tz='UTC')
    active_end   = pd.Timestamp(w['active_end'] + ' 23:59:59', tz='UTC')

    # Active post authors
    active_post_authors = set(
        posts[
            (posts['created_dt'] >= active_start) &
            (posts['created_dt'] <= active_end)
        ]['author']
    )

    # Active comment authors
    active_comment_authors = set(
        comments[
            (comments['created_dt'] >= active_start) &
            (comments['created_dt'] <= active_end)
        ]['author']
    )

    active_users = active_post_authors | active_comment_authors

    # Exposed users this cycle
    exposed_this_cycle = set(exposed_df[exposed_df['cycle'] == cycle]['author'])

    # Unexposed = active but not exposed
    unexposed_authors = active_users - exposed_this_cycle

    for author in unexposed_authors:
        unexposed_records.append({'author': author, 'exposed': False, 'cycle': cycle})

    print(f'Cycle {cycle} — active: {len(active_users):,}, exposed: {len(exposed_this_cycle):,}, unexposed: {len(unexposed_authors):,}')

unexposed_df = pd.DataFrame(unexposed_records)
print(f'\nTotal unexposed: {len(unexposed_df):,}')

Cycle 1 — active: 40,008, exposed: 866, unexposed: 39,165
Cycle 2 — active: 52,292, exposed: 1,176, unexposed: 51,121

Total unexposed: 90,286


## 6) Combine and save exposure labels

In [19]:
exposure_df = pd.concat([exposed_df, unexposed_df], ignore_index=True)

# A user could appear in both cycles — that's fine, keep both rows
print(f'Total exposure records: {len(exposure_df):,}')
print(f'Unique users: {exposure_df["author"].nunique():,}')
print('\nExposed vs unexposed by cycle:')
print(exposure_df.groupby(['cycle', 'exposed']).size().unstack(fill_value=0))

exposure_df.to_parquet(DATA_DIR / 'exposure_labels_v2.parquet', index=False)
print('\nSaved exposure_labels_v2.parquet')

Total exposure records: 92,328
Unique users: 86,829

Exposed vs unexposed by cycle:
exposed  False  True 
cycle                
1        39165    866
2        51121   1176

Saved exposure_labels_v2.parquet


## 7) Quick sanity checks

In [20]:
# Users appearing in both cycles
both_cycles = exposure_df.groupby('author')['cycle'].nunique()
print(f'Users active in both cycles: {(both_cycles == 2).sum():,}')

# Users exposed in both cycles
exposed_both = exposure_df[exposure_df['exposed']].groupby('author')['cycle'].nunique()
print(f'Users exposed in both cycles: {(exposed_both == 2).sum():,}')

# Exposure rate per cycle
for cycle in [1, 2]:
    sub = exposure_df[exposure_df['cycle'] == cycle]
    rate = sub['exposed'].mean()
    print(f'Cycle {cycle} exposure rate: {rate:.2%} ({sub["exposed"].sum():,} exposed / {len(sub):,} active)')

Users active in both cycles: 5,499
Users exposed in both cycles: 41
Cycle 1 exposure rate: 2.16% (866 exposed / 40,031 active)
Cycle 2 exposure rate: 2.25% (1,176 exposed / 52,297 active)
